In [ ]:
# Importe
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pm4py
from scipy.stats import chi2_contingency, mannwhitneyu
import json, re, math, warnings
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 220)
warnings.filterwarnings('ignore', category=FutureWarning)
print('Imports OK')


In [ ]:
# Pfade und Einstellungen
PROJECT_ROOT = Path('..').resolve()
DATA_RAW = PROJECT_ROOT / 'data_raw'
LOG_PATH = DATA_RAW / 'BPI_Challenge_2018.xes.gz'
STEP07_ROOT = PROJECT_ROOT / 'outputs' / 'benchmark_labels_inspection_case_selection'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'benchmark_reconciliation_two_perspectives'
TABLE_DIR = OUTPUT_ROOT / 'tables'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
for d in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)
PRIMARY_YEARS = {2015, 2016}
PRIMARY_LABEL = 'label_scd_p90_or_global'
MIN_GROUP_N = 30
RANDOM_STATE = 42
TECHNICAL_TIE_POLICY = 'timestamp_then_original_row_order'
CASE_FILE_CANDIDATES = [STEP07_ROOT / 'tables' / '32_case_level_benchmark_inspection_core.csv']
CASE_FILE = next((p for p in CASE_FILE_CANDIDATES if p.exists()), None)
if CASE_FILE is None:
    raise FileNotFoundError('Step-07 case export not found. Run notebook 07 first.')
if not LOG_PATH.exists():
    candidates = sorted(DATA_RAW.glob('**/*.xes*'))
    if not candidates:
        raise FileNotFoundError('No XES/XES.GZ found in data_raw')
    LOG_PATH = candidates[0]
print('Project root:', PROJECT_ROOT)
print('Case file:', CASE_FILE)
print('Log path:', LOG_PATH)
print('Output:', OUTPUT_ROOT)


In [ ]:
# Hilfsfunktionen
created_tables, created_figures = ([], [])
quality_gates, analysis_notes = ([], [])

def save_csv(df, name, index=False):
    p = TABLE_DIR / name
    df.to_csv(p, index=index, encoding='utf-8-sig')
    created_tables.append(p)
    return p

def save_json(obj, name):
    p = TABLE_DIR / name
    with open(p, 'w', encoding='utf-8') as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    created_tables.append(p)
    return p

def save_fig(fig, filename):
    p = FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(p, dpi=220, bbox_inches='tight')
    plt.close(fig)
    created_figures.append(p)
    return p

def normalize_name(x):
    return str(x).strip().lower().replace('_', ' ').replace('+', ' ')

def safe_numeric(s):
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors='coerce')
    return pd.to_numeric(s.astype(str).str.replace(',', '.', regex=False).str.strip(), errors='coerce')

def robust_bool(s):
    if s.dtype == bool:
        return s.fillna(False)
    st = s.astype(str).str.strip().str.lower()
    return st.isin(['true', '1', 'yes', 'y', 'ja', 'wahr'])

def resolve_column(columns, base):
    b = normalize_name(base).replace(' ', '')
    scored = []
    for c in columns:
        n = normalize_name(c).replace('case:', '').replace(' ', '')
        if n == b:
            scored.append((0, c))
        elif n.endswith(b):
            scored.append((1, c))
        elif b in n:
            scored.append((2, c))
    return sorted(scored)[0][1] if scored else None

def cliffs_delta(x, y):
    x = pd.to_numeric(pd.Series(x), errors='coerce').dropna().values
    y = pd.to_numeric(pd.Series(y), errors='coerce').dropna().values
    if len(x) == 0 or len(y) == 0:
        return np.nan
    u = mannwhitneyu(x, y, alternative='two-sided').statistic
    return float(2 * u / (len(x) * len(y)) - 1)

def cramers_v(a, b):
    tab = pd.crosstab(a, b)
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return np.nan
    chi2 = chi2_contingency(tab, correction=False)[0]
    n = tab.values.sum()
    denom = min(tab.shape) - 1
    return float(math.sqrt(chi2 / (n * denom))) if n and denom else np.nan

def add_gate(name, status, evidence, consequence):
    quality_gates.append({'gate': name, 'status': status, 'evidence': evidence, 'consequence': consequence})
print('Helpers OK')


In [ ]:
# Fall und Ereignisdaten laden
case_df = pd.read_csv(CASE_FILE)
for c in case_df.columns:
    if c.startswith(('label_', 'has_', 'selected_', 'eligible_')):
        case_df[c] = robust_bool(case_df[c])
for c in ['case_start', 'case_end']:
    if c in case_df.columns:
        case_df[c] = pd.to_datetime(case_df[c], errors='coerce', utc=True)
raw = pm4py.read_xes(str(LOG_PATH))
event_df = raw if isinstance(raw, pd.DataFrame) else pm4py.convert_to_dataframe(raw)
CASE_COL = 'case:concept:name'
TIME_COL = 'time:timestamp'
ACTIVITY_COL = 'concept:name' if 'concept:name' in event_df.columns else 'activity'
if not all((c in event_df.columns for c in [CASE_COL, TIME_COL, ACTIVITY_COL])):
    raise RuntimeError('Core columns missing')
event_df[TIME_COL] = pd.to_datetime(event_df[TIME_COL], errors='coerce', utc=True)
event_df['_event_order'] = np.arange(len(event_df), dtype=np.int64)
event_df['_activity_norm'] = event_df[ACTIVITY_COL].astype(str).str.strip().str.lower()
event_df['_subprocess_norm'] = event_df['subprocess'].astype(str).str.strip().str.lower() if 'subprocess' in event_df.columns else '__missing__'
event_df['_doctype_norm'] = event_df['doctype'].astype(str).str.strip().str.lower() if 'doctype' in event_df.columns else '__missing__'
event_df['_combined_context'] = event_df['_doctype_norm'] + ' | ' + event_df['_subprocess_norm'] + ' | ' + event_df['_activity_norm']
inspection_patterns = 'inspection|on-site|on site|onsite'
event_df['_is_inspection_context'] = event_df['_activity_norm'].str.contains(inspection_patterns, regex=True, na=False) | event_df['_subprocess_norm'].str.contains(inspection_patterns, regex=True, na=False) | event_df['_doctype_norm'].str.contains(inspection_patterns, regex=True, na=False)
event_df['_is_change_objection'] = event_df['_subprocess_norm'].isin(['change', 'objection'])
event_df['_is_begin_payment'] = event_df['_activity_norm'].eq('begin payment')
event_df['_is_abort_payment'] = event_df['_activity_norm'].eq('abort payment')
assert case_df[CASE_COL].nunique() == event_df[CASE_COL].nunique() == 43809
add_gate('Population consistency', 'PASS', f'{case_df[CASE_COL].nunique()} cases in both views', 'Proceed')
print(event_df.shape, case_df.shape)


In [ ]:
# Zahlungsattribute
payment_cols = []
for c in event_df.columns:
    token = normalize_name(c).replace('case:', '').replace(' ', '')
    m = re.search('paymentactual(\\d+)$', token)
    if m:
        payment_cols.append((c, int(m.group(1))))
payment_cols = sorted(payment_cols, key=lambda x: x[1])
if not payment_cols:
    raise RuntimeError('No payment_actual{x} columns found')
pay_case = event_df.groupby(CASE_COL)[[c for c, _ in payment_cols]].first().reset_index()
year_col = resolve_column(event_df.columns, 'year')
if year_col:
    y = event_df.groupby(CASE_COL)[year_col].first().rename('case_year_raw').reset_index()
    pay_case = pay_case.merge(y, on=CASE_COL, how='left')
    pay_case['case_year'] = pd.to_numeric(pay_case['case_year_raw'], errors='coerce').astype('Int64')
else:
    pay_case = pay_case.merge(case_df[[CASE_COL, 'case_year']], on=CASE_COL, how='left')
profile = []
num_cols = []
for c, idx in payment_cols:
    out = f'payment_actual_{idx}_numeric'
    pay_case[out] = safe_numeric(pay_case[c])
    num_cols.append((out, idx, c))
    s = pay_case[out]
    profile.append({'source_column': c, 'index': idx, 'non_missing': int(s.notna().sum()), 'zero': int(s.eq(0).sum()), 'positive': int(s.gt(0).sum()), 'negative': int(s.lt(0).sum()), 'nonzero': int(s.ne(0).fillna(False).sum()), 'min': float(s.min()) if s.notna().any() else np.nan, 'median': float(s.median()) if s.notna().any() else np.nan, 'max': float(s.max()) if s.notna().any() else np.nan})
payment_profile = pd.DataFrame(profile)
save_csv(payment_profile, '01_payment_actual_attribute_profile.csv')
xge1 = [c for c, idx, _ in num_cols if idx >= 1]
pay1 = [c for c, idx, _ in num_cols if idx == 1]
pay_case['additional_official_positive_any_xge1'] = pay_case[xge1].gt(0).any(axis=1) if xge1 else False
pay_case['additional_any_nonzero_xge1'] = pay_case[xge1].fillna(0).ne(0).any(axis=1) if xge1 else False
pay_case['additional_any_negative_xge1'] = pay_case[xge1].lt(0).any(axis=1) if xge1 else False
pay_case['additional_brils_payment1_nonzero'] = pay_case[pay1].fillna(0).ne(0).any(axis=1) if pay1 else False
alt_rows = []
for lab in ['additional_official_positive_any_xge1', 'additional_any_nonzero_xge1', 'additional_any_negative_xge1', 'additional_brils_payment1_nonzero']:
    for pop, mask in [('all_years', pd.Series(True, index=pay_case.index)), ('2015_2016', pay_case.case_year.isin(PRIMARY_YEARS))]:
        alt_rows.append({'label': lab, 'population': pop, 'eligible_cases': int(mask.sum()), 'positive_cases': int(pay_case.loc[mask, lab].sum()), 'prevalence_pct': float(pay_case.loc[mask, lab].mean() * 100)})
additional_sensitivity = pd.DataFrame(alt_rows)
save_csv(additional_sensitivity, '02_additional_payment_definition_sensitivity.csv')
pay_case['additional_definition_group'] = np.select([pay_case.additional_official_positive_any_xge1 & pay_case.additional_brils_payment1_nonzero, pay_case.additional_official_positive_any_xge1 & ~pay_case.additional_brils_payment1_nonzero, ~pay_case.additional_official_positive_any_xge1 & pay_case.additional_brils_payment1_nonzero], ['both', 'official_positive_only', 'brils_nonzero_only'], default='neither')
save_csv(pay_case[[CASE_COL, 'case_year', 'additional_definition_group'] + [c for c, _, _ in num_cols]], '03_additional_payment_case_audit.csv')
brils_prev = float(additional_sensitivity.query("label=='additional_brils_payment1_nonzero' and population=='all_years'").prevalence_pct.iloc[0])
official_prev = float(additional_sensitivity.query("label=='additional_official_positive_any_xge1' and population=='all_years'").prevalence_pct.iloc[0])
status = 'PASS' if abs(brils_prev - 5.0) <= 1.0 else 'WARN'
add_gate('Additional payment reconciliation', status, f'Brils-rule={brils_prev:.2f} %, official-positive={official_prev:.2f} %', 'Vor Verwendung Vorzeichen und Feldunterschiede prüfen')
display(payment_profile)
display(additional_sensitivity)


In [ ]:
# Verspätete Zahlungen
required = ['label_late_payment_benchmark_primary', 'label_late_payment_official_stable', 'eligible_late_payment_primary', 'label_reopened_official', PRIMARY_LABEL]
missing = [c for c in required if c not in case_df.columns]
if missing:
    raise RuntimeError(f'Missing Step-07 labels: {missing}')
eligible = case_df.eligible_late_payment_primary
case_df['late_definition_group'] = np.select([eligible & case_df.label_late_payment_benchmark_primary & case_df.label_late_payment_official_stable, eligible & case_df.label_late_payment_benchmark_primary & ~case_df.label_late_payment_official_stable, eligible & ~case_df.label_late_payment_benchmark_primary & case_df.label_late_payment_official_stable, eligible & ~case_df.label_late_payment_benchmark_primary & ~case_df.label_late_payment_official_stable], ['both_late', 'brils_only', 'official_only', 'neither'], default='ineligible')
late_group_summary = case_df.groupby('late_definition_group').agg(n_cases=(CASE_COL, 'size'), scd_rate=(PRIMARY_LABEL, 'mean'), reopened_rate=('label_reopened_official', 'mean'), additional_rate=('label_additional_payment_any', 'mean'), median_duration=('duration_days', 'median')).reset_index()
for c in ['scd_rate', 'reopened_rate', 'additional_rate']:
    late_group_summary[c] *= 100
save_csv(late_group_summary, '04_late_definition_group_summary.csv')
relevant = event_df[event_df._is_begin_payment | event_df._is_abort_payment | event_df._is_change_objection].copy()
relevant = relevant.merge(case_df[[CASE_COL, 'case_year', 'late_definition_group']], on=CASE_COL, how='left')
relevant = relevant.sort_values([CASE_COL, TIME_COL, '_event_order'], kind='mergesort')
relevant['event_calendar_year'] = relevant[TIME_COL].dt.year

def timeline_features(g):
    cy = pd.to_numeric(g.case_year.iloc[0], errors='coerce')
    begins = g[g._is_begin_payment]
    aborts = g[g._is_abort_payment]
    timely = begins[begins[TIME_COL].dt.year <= cy] if pd.notna(cy) else begins.iloc[0:0]
    first_timely = timely[TIME_COL].min() if len(timely) else pd.NaT
    after_first = g[g[TIME_COL] > first_timely] if pd.notna(first_timely) else g.iloc[0:0]
    return pd.Series({'begin_count': len(begins), 'abort_count': len(aborts), 'timely_begin_count': len(timely), 'first_begin': begins[TIME_COL].min() if len(begins) else pd.NaT, 'last_begin': begins[TIME_COL].max() if len(begins) else pd.NaT, 'first_abort': aborts[TIME_COL].min() if len(aborts) else pd.NaT, 'last_abort': aborts[TIME_COL].max() if len(aborts) else pd.NaT, 'has_change_objection_after_first_timely_begin': bool(after_first._is_change_objection.any()) if len(after_first) else False, 'has_abort_after_first_timely_begin': bool(after_first._is_abort_payment.any()) if len(after_first) else False, 'has_begin_after_case_year': bool((begins[TIME_COL].dt.year > cy).any()) if pd.notna(cy) and len(begins) else False})
late_timeline = relevant.groupby(CASE_COL, group_keys=False).apply(timeline_features).reset_index()
late_timeline = case_df[[CASE_COL, 'case_year', 'late_definition_group', 'label_reopened_official', PRIMARY_LABEL]].merge(late_timeline, on=CASE_COL, how='left')
save_csv(late_timeline, '05_late_payment_timeline_diagnostics.csv')
rng = np.random.default_rng(RANDOM_STATE)
sample_ids = []
for grp, n in [('brils_only', 20), ('both_late', 15), ('neither', 10)]:
    ids = case_df.loc[case_df.late_definition_group.eq(grp), CASE_COL].astype(str).values
    if len(ids):
        sample_ids.extend(rng.choice(ids, size=min(n, len(ids)), replace=False).tolist())
trace_export = relevant[relevant[CASE_COL].astype(str).isin(set(sample_ids))][[CASE_COL, TIME_COL, '_event_order', '_activity_norm', '_subprocess_norm', '_doctype_norm', 'late_definition_group']]
save_csv(trace_export, '06_representative_payment_change_timelines.csv')
brils_only = late_timeline[late_timeline.late_definition_group.eq('brils_only')]
share_timely = float((brils_only.timely_begin_count.fillna(0) > 0).mean() * 100) if len(brils_only) else np.nan
share_reopened = float(brils_only.label_reopened_official.mean() * 100) if len(brils_only) else np.nan
add_gate('Late payment reconciliation', 'WARN', f'Brils-only cases={len(brils_only)}; {share_timely:.1f} % have a timely begin; {share_reopened:.1f} % reopened', 'Keep Brils-style only as literature replication until trace review')
display(late_group_summary)


In [ ]:
# Reopened und Zensierung
co = event_df[event_df._is_change_objection].groupby(CASE_COL)[TIME_COL].min().rename('first_reopen_time').reset_index()
censor = case_df[[CASE_COL, 'case_year', 'case_start', 'case_end', 'label_reopened_official']].merge(co, on=CASE_COL, how='left')
censor['days_to_reopen'] = (censor.first_reopen_time - censor.case_start).dt.total_seconds() / 86400
log_end = event_df[TIME_COL].max()
censor['max_observable_days_from_start'] = (log_end - censor.case_start).dt.total_seconds() / 86400
for horizon in [90, 180, 270, 365, 540]:
    censor[f'reopened_within_{horizon}d'] = censor.days_to_reopen.le(horizon).fillna(False)
rows = []
for year, g in censor.groupby('case_year'):
    row = {'case_year': year, 'n_cases': len(g), 'median_max_observable_days': g.max_observable_days_from_start.median(), 'reopened_eventually_pct': g.label_reopened_official.mean() * 100}
    for h in [90, 180, 270, 365, 540]:
        row[f'reopened_within_{h}d_pct'] = g[f'reopened_within_{h}d'].mean() * 100
    rows.append(row)
censor_summary = pd.DataFrame(rows)
save_csv(censor_summary, '07_reopened_equal_followup_by_year.csv')
save_csv(censor[[CASE_COL, 'case_year', 'days_to_reopen', 'max_observable_days_from_start']], '08_reopened_timing_case_level.csv')
med17 = censor_summary.loc[censor_summary.case_year.eq(2017), 'median_max_observable_days']
med16 = censor_summary.loc[censor_summary.case_year.eq(2016), 'median_max_observable_days']
if len(med17) and len(med16) and (med17.iloc[0] < med16.iloc[0] - 180):
    add_gate('2017 benchmark censoring', 'WARN', f'Median observable horizon 2017={med17.iloc[0]:.0f}d vs 2016={med16.iloc[0]:.0f}d', 'Use 2015/2016 primary; 2017 sensitivity only')
else:
    add_gate('2017 benchmark censoring', 'REVIEW', 'Equal-follow-up comparison required', 'Do not infer improvement from raw yearly rates')
display(censor_summary)


In [ ]:
# Inspection bereinigtes SCD
noninsp = event_df[~event_df._is_inspection_context].copy()
noninsp_counts = noninsp.groupby(CASE_COL).size().rename('event_count_no_inspection')
ctx_counts = noninsp.groupby([CASE_COL, '_combined_context'], observed=True).size().rename('n').reset_index()
ctx_counts['extra'] = (ctx_counts.n - 1).clip(lower=0)
noninsp_rework = ctx_counts.groupby(CASE_COL).extra.sum().rename('rework_no_inspection')
residual = pd.concat([noninsp_counts, noninsp_rework], axis=1).fillna(0).reset_index()
case_df = case_df.merge(residual, on=CASE_COL, how='left')
case_df[['event_count_no_inspection', 'rework_no_inspection']] = case_df[['event_count_no_inspection', 'rework_no_inspection']].fillna(0)
th_event = float(case_df.event_count_no_inspection.quantile(0.9))
th_rework = float(case_df.rework_no_inspection.quantile(0.9))
case_df['label_scd_no_inspection_p90_or'] = (case_df.event_count_no_inspection >= th_event) | (case_df.rework_no_inspection >= th_rework)
case_df['scd_inspection_adjustment_group'] = np.select([case_df[PRIMARY_LABEL] & case_df.label_scd_no_inspection_p90_or, case_df[PRIMARY_LABEL] & ~case_df.label_scd_no_inspection_p90_or, ~case_df[PRIMARY_LABEL] & case_df.label_scd_no_inspection_p90_or], ['both', 'original_only_inspection_burden', 'residual_only'], default='neither')
summary = case_df.groupby('scd_inspection_adjustment_group').agg(n_cases=(CASE_COL, 'size'), inspection_rate=('has_inspection_event_context', 'mean'), median_duration=('duration_days', 'median'), median_event_no_inspection=('event_count_no_inspection', 'median'), median_rework_no_inspection=('rework_no_inspection', 'median')).reset_index()
summary.inspection_rate *= 100
save_csv(summary, '09_scd_inspection_adjustment_summary.csv')
save_json({'event_count_no_inspection_p90': th_event, 'rework_no_inspection_p90': th_rework}, '10_scd_no_inspection_thresholds.json')
rows = []
for flag in ['selected_random', 'selected_risk', 'selected_manually', 'selected_any_inspection', 'has_inspection_event_context']:
    for val, g in case_df.groupby(flag):
        rows.append({'flag': flag, 'value': bool(val), 'n_cases': len(g), 'original_scd_pct': g[PRIMARY_LABEL].mean() * 100, 'residual_scd_pct': g.label_scd_no_inspection_p90_or.mean() * 100, 'median_noninspection_events': g.event_count_no_inspection.median(), 'median_noninspection_rework': g.rework_no_inspection.median()})
inspection_adjusted = pd.DataFrame(rows)
save_csv(inspection_adjusted, '11_inspection_flags_original_vs_residual_scd.csv')
add_gate('Inspection mechanical-burden control', 'PASS', f'Residual SCD created with thresholds {th_event:.1f}/{th_rework:.1f}', 'Use residual analysis to distinguish required inspection work from broader complexity')
display(summary)
display(inspection_adjusted)


In [ ]:
# Inspection Perspektive
case_df['inspection_scd_group'] = np.select([case_df[PRIMARY_LABEL] & case_df.has_inspection_event_context, case_df[PRIMARY_LABEL] & ~case_df.has_inspection_event_context, ~case_df[PRIMARY_LABEL] & case_df.has_inspection_event_context], ['SCD_inspection', 'SCD_no_inspection', 'nonSCD_inspection'], default='neither')
metrics = ['event_count', 'combined_rework_extra', 'event_count_no_inspection', 'rework_no_inspection', 'duration_days', 'n_documents', 'n_subprocesses', 'n_resources']
rows = []
for grp, g in case_df.groupby('inspection_scd_group'):
    for m in metrics:
        rows.append({'group': grp, 'metric': m, 'n': len(g), 'median': g[m].median(), 'mean': g[m].mean()})
inspection_deep = pd.DataFrame(rows)
save_csv(inspection_deep, '12_inspection_perspective_metrics.csv')

def presence_difference(events, positive_cases, negative_cases, value_col, perspective, top_n=40):
    e = events[[CASE_COL, value_col]].drop_duplicates()
    pos = e[e[CASE_COL].isin(positive_cases)][value_col].value_counts()
    neg = e[e[CASE_COL].isin(negative_cases)][value_col].value_counts()
    vals = pos.index.union(neg.index)
    out = []
    for v in vals:
        pp = 100 * pos.get(v, 0) / max(len(positive_cases), 1)
        npct = 100 * neg.get(v, 0) / max(len(negative_cases), 1)
        out.append({'perspective': perspective, 'signature_level': value_col, 'value': v, 'positive_presence_pct': pp, 'negative_presence_pct': npct, 'difference_pp': pp - npct, 'abs_difference_pp': abs(pp - npct)})
    return pd.DataFrame(out).sort_values('abs_difference_pp', ascending=False).head(top_n)
scd_ins = set(case_df.loc[case_df.inspection_scd_group.eq('SCD_inspection'), CASE_COL])
scd_non = set(case_df.loc[case_df.inspection_scd_group.eq('SCD_no_inspection'), CASE_COL])
noninsp_events = event_df[~event_df._is_inspection_context]
sigs = []
for col in ['_activity_norm', '_subprocess_norm', '_doctype_norm', '_combined_context']:
    sigs.append(presence_difference(noninsp_events, scd_ins, scd_non, col, 'inspection_driven_residual'))
inspection_sigs = pd.concat(sigs, ignore_index=True)
save_csv(inspection_sigs, '13_inspection_residual_process_signatures.csv')


In [ ]:
# Change und Objection Perspektive
case_df['reopened_type_clean'] = case_df.get('reopened_type', 'none').fillna('none')
primary_mask = case_df.case_year.isin(PRIMARY_YEARS)
rows = []
for pop, mask in [('all_years', pd.Series(True, index=case_df.index)), ('2015_2016', primary_mask)]:
    for grp, g in case_df[mask].groupby('reopened_type_clean'):
        rows.append({'population': pop, 'reopened_type': grp, 'n_cases': len(g), 'scd_rate_pct': g[PRIMARY_LABEL].mean() * 100, 'median_duration': g.duration_days.median(), 'median_events': g.event_count.median(), 'median_rework': g.combined_rework_extra.median(), 'additional_payment_pct': g.label_additional_payment_any.mean() * 100, 'late_payment_pct': g.loc[g.eligible_late_payment_primary, 'label_late_payment_benchmark_primary'].mean() * 100 if g.eligible_late_payment_primary.any() else np.nan})
reopened_deep = pd.DataFrame(rows)
save_csv(reopened_deep, '14_change_objection_type_metrics.csv')
scd_rep = set(case_df.loc[case_df[PRIMARY_LABEL] & case_df.label_reopened_official & primary_mask, CASE_COL])
scd_norep = set(case_df.loc[case_df[PRIMARY_LABEL] & ~case_df.label_reopened_official & primary_mask, CASE_COL])
nondef_events = event_df[~event_df._is_change_objection]
sigs = []
for col in ['_activity_norm', '_subprocess_norm', '_doctype_norm', '_combined_context']:
    sigs.append(presence_difference(nondef_events, scd_rep, scd_norep, col, 'change_objection_residual'))
reopened_sigs = pd.concat(sigs, ignore_index=True)
save_csv(reopened_sigs, '15_change_objection_residual_process_signatures.csv')
add_gate('Two perspective selection', 'PASS', 'Inspection-driven and Change/Objection-driven perspectives analysed separately', 'Carry both into final descriptive chapter')
display(reopened_deep)


In [ ]:
# Qualitätsprüfungen
quality_df = pd.DataFrame(quality_gates)
save_csv(quality_df, '17_quality_gate_register.csv')
display(quality_df)


In [ ]:
# Abbildungen
fig, ax = plt.subplots(figsize=(9, 5))
p = additional_sensitivity[additional_sensitivity.population.eq('all_years')].sort_values('prevalence_pct')
ax.barh(p.label, p.prevalence_pct)
ax.set_xlabel('Anteil (%)')
ax.set_title('Additional Payment: Definitionssensitivitaet')
ax.grid(axis='x', alpha=0.3)
save_fig(fig, 'fig_01_additional_payment_sensitivity.png')
fig, ax = plt.subplots(figsize=(8, 5))
p = late_group_summary[late_group_summary.late_definition_group.ne('ineligible')].sort_values('n_cases')
ax.barh(p.late_definition_group, p.n_cases)
ax.set_xlabel('Cases 2015/2016')
ax.set_title('Late-Payment definition disagreement')
ax.grid(axis='x', alpha=0.3)
save_fig(fig, 'fig_02_late_definition_groups.png')
fig, ax = plt.subplots(figsize=(9, 5))
for h in [180, 270, 365]:
    ax.plot(censor_summary.case_year.astype(str), censor_summary[f'reopened_within_{h}d_pct'], marker='o', label=f'within {h}d')
ax.set_ylabel('Reopened rate (%)')
ax.set_title('Reopened at equal follow-up horizons')
ax.legend()
ax.grid(alpha=0.3)
save_fig(fig, 'fig_03_reopened_equal_followup.png')
fig, ax = plt.subplots(figsize=(9, 5))
p = inspection_adjusted[inspection_adjusted.flag.isin(['selected_any_inspection', 'has_inspection_event_context'])]
x = np.arange(len(p))
w = 0.36
ax.bar(x - w / 2, p.original_scd_pct, w, label='Original SCD')
ax.bar(x + w / 2, p.residual_scd_pct, w, label='SCD without inspection events')
ax.set_xticks(x)
ax.set_xticklabels(p.flag + '=' + p.value.astype(str), rotation=20, ha='right')
ax.set_ylabel('Rate (%)')
ax.set_title('Mechanical inspection burden control')
ax.legend()
ax.grid(axis='y', alpha=0.3)
save_fig(fig, 'fig_04_original_vs_residual_scd.png')
fig, ax = plt.subplots(figsize=(8, 5))
vals = [case_df.loc[case_df.inspection_scd_group.eq('SCD_inspection'), 'duration_days'].median(), case_df.loc[case_df.inspection_scd_group.eq('SCD_no_inspection'), 'duration_days'].median(), case_df.loc[case_df[PRIMARY_LABEL] & case_df.label_reopened_official & primary_mask, 'duration_days'].median(), case_df.loc[case_df[PRIMARY_LABEL] & ~case_df.label_reopened_official & primary_mask, 'duration_days'].median()]
labs = ['SCD + Inspection', 'SCD without Inspection', 'SCD + Reopened', 'SCD without Reopened']
ax.barh(labs, vals)
ax.set_xlabel('Median duration (days)')
ax.set_title('Two mechanisms inside SCD')
ax.grid(axis='x', alpha=0.3)
save_fig(fig, 'fig_05_two_scd_mechanisms.png')
print('Figures:', len(created_figures))


In [ ]:
# Falldaten speichern
core_cols = [CASE_COL, 'case_year', 'case_department', PRIMARY_LABEL, 'label_scd_no_inspection_p90_or', 'scd_inspection_adjustment_group', 'inspection_scd_group', 'label_reopened_official', 'reopened_type_clean', 'label_late_payment_benchmark_primary', 'label_late_payment_official_stable', 'late_definition_group', 'selected_random', 'selected_risk', 'selected_manually', 'selected_any_inspection', 'has_inspection_event_context', 'event_count', 'combined_rework_extra', 'event_count_no_inspection', 'rework_no_inspection', 'duration_days']
save_csv(case_df[[c for c in core_cols if c in case_df.columns]], '19_case_level_reconciliation_core.csv')
